# Tier 4.0 — LLM-as-a-Judge Re-Ranking with Hybrid First Stage (Azure GPU)

**BSARD RAG Thesis | RQ1 | T4.0-Hybrid Experiment**

Experiment: `llm_rerank_binary_top50_hybrid_rrf_k60_test`
First stage: `hybrid_rrf_k60` — BM25 (k1=1.5, b=0.25, lemmatize, text_only) + mE5-large concat_2x, RRF k=60, first_stage_k=100
Identical to `azure_tier40_llm_rerank.ipynb` except for the first-stage retriever.

## Before running — one-time setup

1. **GPU compute instance**: Azure ML → Compute → Create → `Standard_NC6ads_A10_v4` (preferred) or `NC4as_T4_v3`
2. **Set Cell 0** with your `GITHUB_TOKEN` and `AZURE_CONTAINER_SAS_URL`
3. **Prerequisites in blob storage** (upload these locally before running):
   - `embeddings/intfloat_multilingual_e5_large_concat_2x.npy` (~87 MB)
   - `embeddings/intfloat_multilingual_e5_large_concat_2x_ids.npy` (~0.2 MB)
   - `results/hybrid/hybrid_rrf_k60_test.json` (significance anchor)
   - `llm_judge_cache_binary_test_tok1000.json` (optional, for cache reuse)
4. Run cells top to bottom

## Interrupt recovery

Cell 11 is fully resumable:
- **Score cache** (`llm_judge_cache_binary_test_tok1000.json`) is checkpointed every 20 questions — LLM calls for scored pairs are never repeated.
- **Per-question checkpoint** (`llm_rerank_binary_top50_hybrid_rrf_k60_test.ckpt.json`) stores the final article ranking per question — the first-stage retrieval is also skipped on resume.
- On restart: re-run Cell 11 from the top. It detects the checkpoint and resumes from the last completed question.

## Ablation design

| Comparison | Measures |
|---|---|
| T4.0-hybrid vs `hybrid_rrf_k60` (T3-A) | Value of LLM re-ranking on hybrid pool |
| T4.0-hybrid vs T4.0-BM25 | Pool quality effect (hybrid vs BM25 first stage) |
| T4.2-hybrid vs T4.0-hybrid | Pure ReAct loop value on hybrid pool |

## Expected execution times (T4 GPU)

| Phase | Expected time |
|---|---|
| Setup (Cells 0–7) | ~25 min (mE5-large load adds ~3 min vs BM25-only) |
| TEST experiment (Cell 11) | ~2–3h worst case; less with cache reuse from prior runs |
| Significance tests (Cell 12) | ~5 min |
| Upload to blob (Cell 14) | ~1 min |

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────────────────
# Set GITHUB_TOKEN and AZURE_CONTAINER_SAS_URL before running any other cell.

GITHUB_TOKEN = ''
# How to get:
#   github.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → New token → scope: repo → Generate

AZURE_CONTAINER_SAS_URL = ''
# How to get:
#   Azure Portal → Storage Accounts → your account
#   → Containers → bsard-data → (...) → Generate SAS
#   → Permissions: Read + List + Write → Expiry: 1 year → Generate
#   → Copy the full "Blob SAS URL" (starts with https://...)

REPO     = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root

assert GITHUB_TOKEN,            'Set GITHUB_TOKEN above before running!'
assert AZURE_CONTAINER_SAS_URL, 'Set AZURE_CONTAINER_SAS_URL above before running!'
print('Config OK')

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('GPU:', result.stdout.strip())
    print('GPU OK')
else:
    print('WARNING: No GPU detected!')
    print('  Azure ML: ensure compute instance uses NC4as_T4_v3 or NC6ads_A10_v4')
    print(result.stderr)

In [ ]:
# ── Cell 2: Install Ollama and pull llama3.1:8b (~10 min on first run) ────────
import json, os, subprocess, time, urllib.request

OLLAMA_LOG = '/tmp/ollama_server.log'

def model_available() -> bool:
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            return any('llama3.1' in m['name'] for m in json.loads(r.read()).get('models', []))
    except Exception:
        return False

if not os.path.exists('/usr/local/bin/ollama'):
    print('Installing Ollama via official script...')
    subprocess.run(
        ['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True
    )
    print('Ollama installed.')
else:
    print('Ollama already installed.')

if not model_available():
    print('Starting Ollama server...')
    subprocess.Popen(
        ['ollama', 'serve'],
        env={
            **os.environ,
            'HOME': '/root',
            'OLLAMA_NUM_GPU': '99',
            'OLLAMA_FLASH_ATTENTION': '1',
            'OLLAMA_HOST': '0.0.0.0:11434',
        },
        stdout=open(OLLAMA_LOG, 'w'),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)
    if not model_available():
        print('Pulling llama3.1:8b (~4.7 GB, ~5-10 min)...')
        subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)
        time.sleep(3)

resp   = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=10)
models = [m['name'] for m in json.loads(resp.read()).get('models', [])]
print('Available models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'
print('Ollama ready.')

In [ ]:
# ── Cell 3: Download data from Azure Blob Storage ─────────────────────────────
# Supports resume of partial downloads (picks up from byte offset).
# Missing optional blobs (cache, significance anchors) show a warning, not an error.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob', 'tqdm'],
               check=True)

from azure.storage.blob import ContainerClient
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_DIR  = Path(REPO_DIR) / 'output'
EMB_DIR     = OUTPUT_DIR / 'embeddings'
RESULTS_DIR = OUTPUT_DIR / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'hybrid').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'agentic' / 'llm_judge' / 'llm_rerank').mkdir(parents=True, exist_ok=True)

EMB_SLUG = 'intfloat_multilingual_e5_large_concat_2x'
client   = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)


def _download_blob(blob_name: str, dest_path: Path, required: bool = True) -> bool:
    """Download blob → dest_path with tqdm progress and resume support."""
    try:
        bc         = client.get_blob_client(blob_name)
        total_size = bc.get_blob_properties()['size']
    except Exception:
        tag = '[REQUIRED]' if required else '[optional]'
        print(f'  {tag} {blob_name} — not found in blob storage')
        if required:
            raise FileNotFoundError(f'Required blob missing: {blob_name}')
        return False

    existing = dest_path.stat().st_size if dest_path.exists() else 0
    if existing == total_size:
        print(f'  Already complete: {dest_path.name} ({total_size / 1e6:.1f} MB)')
        return True

    offset = existing if 0 < existing < total_size else 0
    mode   = 'ab' if offset > 0 else 'wb'
    if offset > 0:
        print(f'  Resuming {dest_path.name}: {offset/1e6:.1f}/{total_size/1e6:.1f} MB already done')

    with tqdm(total=total_size, initial=offset, unit='B', unit_scale=True,
              desc=f'  {dest_path.name}', ncols=90, leave=True) as pbar:
        with open(dest_path, mode) as f:
            stream = bc.download_blob(offset=offset)
            for chunk in stream.chunks():
                f.write(chunk)
                pbar.update(len(chunk))
    return True


# ── Required files ─────────────────────────────────────────────────────────────
print('=== Required files ===')
_download_blob('bsard_articles_dedup.parquet', OUTPUT_DIR / 'bsard_articles_dedup.parquet')
_download_blob('bsard_corpus.db',              OUTPUT_DIR / 'bsard_corpus.db')
_download_blob(f'embeddings/{EMB_SLUG}.npy',      EMB_DIR / f'{EMB_SLUG}.npy')
_download_blob(f'embeddings/{EMB_SLUG}_ids.npy',  EMB_DIR / f'{EMB_SLUG}_ids.npy')

# ── Optional files (score cache + significance anchors) ────────────────────────
print('\n=== Optional files (warnings only if missing) ===')
_cache_ok = _download_blob(
    'llm_judge_cache_binary_test_tok1000.json',
    OUTPUT_DIR / 'llm_judge_cache_binary_test_tok1000.json',
    required=False,
)
if not _cache_ok:
    print('  → Cache will start empty — all 11,100 LLM calls will be fresh (2-3h)')

_download_blob(
    'results/hybrid/hybrid_rrf_k60_test.json',
    RESULTS_DIR / 'hybrid' / 'hybrid_rrf_k60_test.json',
    required=False,
)
_download_blob(
    'results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top50_test.json',
    RESULTS_DIR / 'agentic' / 'llm_judge' / 'llm_rerank' / 'llm_rerank_binary_top50_test.json',
    required=False,
)

print('\nAll downloads complete.')

In [ ]:
# ── Cell 4: Clone GitHub repo ─────────────────────────────────────────────────
import os, subprocess

if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest...')
    subprocess.run(
        ['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin',
         f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'],
        capture_output=True, text=True
    )
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f'Cloning {REPO}...')
    r = subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError('git clone failed')

os.chdir(REPO_DIR)
branch = subprocess.run(['git', 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print(f'Working directory: {os.getcwd()}  |  branch: {branch}')

In [ ]:
# ── Cell 5: Install Python dependencies (~5 min) ──────────────────────────────
import subprocess, sys, os

cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
cuda_ver = 'cu118' if 'release 11' in cuda_out else 'cu121'
print(f'CUDA detected → torch variant: {cuda_ver}')

cmds = [
    ([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
     'requirements.txt'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'],
     'azure-storage-blob'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
      '--index-url', f'https://download.pytorch.org/whl/{cuda_ver}'],
     'torch'),
    ([sys.executable, '-m', 'pip', 'install', '-q',
      'langgraph>=0.1.0', 'langchain-core>=0.2.0', 'requests'],
     'langgraph + langchain-core + requests'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'tf-keras'],
     'tf-keras'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.2'],
     'timm>=0.9.2'),
]
for cmd, label in cmds:
    print(f'  {label} ...', end='', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(' OK' if r.returncode == 0 else f' WARN({r.returncode})')
    if r.returncode != 0:
        print(r.stderr[-200:])

# bsard_evaluation — editable local package
# bsard_evaluation lives in the same mono-repo (RQ3_Autonomous_Evaluation),
# already cloned above — just install it editable.
RQ3_DIR = f'{CLONE_DIR}/RQ3_Autonomous_Evaluation'
print('  bsard_evaluation ...', end='', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', RQ3_DIR],
                   capture_output=True, text=True)
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

# spaCy French model
import spacy as _spacy
_sv = _spacy.__version__
_base = 'https://github.com/explosion/spacy-models/releases/download'
_whl  = f'fr_core_news_lg-{_sv}/fr_core_news_lg-{_sv}-py3-none-any.whl'
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl}'],
    capture_output=True, text=True
)
if r.returncode != 0:
    _sv2 = '.'.join(_sv.split('.')[:2]) + '.0'
    _whl2 = f'fr_core_news_lg-{_sv2}/fr_core_news_lg-{_sv2}-py3-none-any.whl'
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl2}'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'spaCy fr_core_news_lg install failed:\n{r.stderr[-400:]}')

import spacy
spacy.load('fr_core_news_lg')
print(f'spaCy {spacy.__version__} OK — fr_core_news_lg loaded')

In [ ]:
# ── Cell 6: Pre-flight checks ─────────────────────────────────────────────────
import json, os, sys, subprocess
import requests as _req
from pathlib import Path

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Ollama
r = _req.get('http://localhost:11434/api/tags', timeout=5)
models = [m['name'] for m in r.json().get('models', [])]
print('Ollama models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'

# Fewshot examples
assert Path('evaluation/data/fewshot_examples.json').exists(), 'fewshot_examples.json missing!'
print('fewshot_examples.json: OK')

# Corpus files
for f in ['output/bsard_articles_dedup.parquet', 'output/bsard_corpus.db']:
    assert Path(f).exists(), f'{f} missing!'
    print(f'  {f}: OK')

# mE5-large embeddings
EMB_SLUG = 'intfloat_multilingual_e5_large_concat_2x'
emb_path = Path(f'output/embeddings/{EMB_SLUG}.npy')
ids_path = Path(f'output/embeddings/{EMB_SLUG}_ids.npy')
if emb_path.exists() and ids_path.exists():
    print(f'  Embeddings: {emb_path.stat().st_size/1e6:.1f} MB — will load from disk (fast)')
else:
    print(f'  [WARN] Embeddings not found — DenseRetriever will encode from scratch (~10 min on T4)')

# Score cache
cache_path = Path('output/llm_judge_cache_binary_test_tok1000.json')
if cache_path.exists():
    cache_size = len(json.loads(cache_path.read_text(encoding='utf-8')))
    print(f'  Score cache: {cache_size:,} cached pairs → significant LLM speedup expected')
else:
    print(f'  [WARN] Score cache missing → all LLM calls fresh (worst case ~2-3h)')

# Prior checkpoint
ckpt_glob = list(Path('output/results/agentic/llm_judge/llm_rerank').glob('*.ckpt.json'))
if ckpt_glob:
    for cp in ckpt_glob:
        n = len(json.loads(cp.read_text(encoding='utf-8')))
        print(f'  Checkpoint found: {cp.name} ({n}/222 questions done) — Cell 11 will resume')

# GPU VRAM
gpu_mem = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.used,memory.free,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'\nVRAM: {gpu_mem}')
print('\nAll checks passed.')

In [ ]:
# ── Cell 7: LLM latency benchmark — confirm GPU speed ─────────────────────────
import time, os, sys
import requests as _req

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from retrieval.agentic.llm_eval_prompts import (
    load_fewshot_examples, format_fewshot_block, LLM_JUDGE_BINARY_PROMPT
)

def _benchmark(article_words: int, label: str) -> float:
    examples = load_fewshot_examples()
    fewshot  = format_fewshot_block(examples, 'binary')
    prompt   = LLM_JUDGE_BINARY_PROMPT.format(
        fewshot_block=fewshot,
        question='Quelle est la peine pour vol simple ?',
        article_text_truncated=' '.join(['mot'] * article_words),
    )
    t0  = time.perf_counter()
    r   = _req.post('http://localhost:11434/api/generate',
                    json={'model': 'llama3.1:8b', 'prompt': prompt, 'stream': False,
                          'options': {'temperature': 0.0, 'num_predict': 4}},
                    timeout=300).json()
    elapsed = time.perf_counter() - t0
    n_in    = r.get('prompt_eval_count', 0)
    print(f'  [{label}] {article_words} words → {n_in} tokens | '
          f'{elapsed:.2f}s | {n_in/elapsed:.0f} tok/s | response={r["response"]!r}')
    return elapsed

print('Benchmarking (warmup + 2 lengths)...')
_benchmark(200, 'warmup')
e_300  = _benchmark(300,  '300-tok')
e_1000 = _benchmark(1000, '1000-tok')

print('\n--- Time estimates for TEST experiment (222 questions × top_n=50) ---')
# Worst case: 0% cache hit rate
for label, e in [('300-tok  top-50', e_300), ('1000-tok top-50', e_1000)]:
    h_worst = 222 * 50 * e / 3600
    print(f'  {label}: ~{h_worst:.1f}h worst case (0% cache) | '
          f'~{h_worst * 0.3:.1f}h if 70% cache hits')
print('\nNote: cache key = (question_id, article_id, prompt_variant, max_article_tokens)')
print('      Cache is first-stage agnostic — hits from both BM25 and prior hybrid runs.')

## Cell 8: Val experiments — not run

Val experiments are **not run**.
Hyperparameters (`BEST_TOP_N=50`, `BEST_VARIANT='binary'`) are fixed a priori per plan §3.2.
Proceed directly to Cell 9 to confirm the canonical config.

In [ ]:
# ── Cell 9: Canonical hyperparameters (fixed a priori — plan §3.2) ────────────
BEST_TOP_N   = 50       # pre-specified: good coverage without scoring low-probability candidates
BEST_VARIANT = 'binary' # pre-specified: 8B-class models produce poorly calibrated continuous scores
print(f'Canonical config: variant={BEST_VARIANT}  top_n={BEST_TOP_N}')

In [ ]:
# ── Cell 10: Build HybridRetriever ────────────────────────────────────────────
# BM25 (k1=1.5, b=0.25, lemmatize, text_only) + mE5-large (concat_2x), RRF k=60
# DenseRetriever auto-loads from .npy if present; falls back to full GPU encode (~10 min).
# BM25 loads from disk cache if tokenization was run before; otherwise tokenizes (~2-5 min).
import os, sys, time, subprocess
from pathlib import Path
import pandas as pd

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from retrieval.sparse import BM25Retriever
from retrieval.dense import DenseRetriever
from retrieval.hybrid import HybridRetriever

def _vram():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    return r.stdout.strip()

print('Loading corpus...')
corpus = pd.read_parquet('output/bsard_articles_dedup.parquet')
print(f'  {len(corpus):,} articles')

# ── BM25 ──────────────────────────────────────────────────────────────────────
print('\nBuilding BM25Retriever (lemmatize, text_only, k1=1.5, b=0.25)...')
t0     = time.perf_counter()
sparse = BM25Retriever(
    corpus, variant='okapi', normalization='lemmatize',
    field_weighting='text_only', k1=1.5, b=0.25,
)
bm25_s = time.perf_counter() - t0
print(f'  BM25 ready in {bm25_s:.1f}s')

# ── DenseRetriever ────────────────────────────────────────────────────────────
print('\nBuilding DenseRetriever (mE5-large, concat_2x)...')
print(f'  VRAM before: {_vram()}')
t0    = time.perf_counter()
dense = DenseRetriever(
    corpus,
    model_name='intfloat/multilingual-e5-large',
    field_weighting='concat_2x',
    passage_prefix='passage: ',
    query_prefix='query: ',
    device='cuda',
    embeddings_dir=Path('output/embeddings'),
)
dense_s    = time.perf_counter() - t0
build_src  = dense._index_build_source  # 'disk' | 'encode' | 'cache'
print(f'  DenseRetriever ready in {dense_s:.1f}s  (source: {build_src})')
print(f'  VRAM after:  {_vram()}')
if build_src == 'encode':
    print('  [INFO] Embeddings encoded from scratch — will be saved to output/embeddings/ for next run')

# ── HybridRetriever ───────────────────────────────────────────────────────────
print('\nBuilding HybridRetriever (RRF k=60, first_stage_k=100)...')
hybrid_retriever = HybridRetriever(
    sparse_retriever=sparse,
    dense_retriever=dense,
    fusion_method='rrf',
    rrf_k=60,
    first_stage_k=100,
)
print('  HybridRetriever ready.')

# Sanity check (also warms up the GPU — mE5-large loads model lazily on first encode)
print('\nRunning sanity check (warm-up query)...')
t0 = time.perf_counter()
_ids, _lat = hybrid_retriever.retrieve('Quelle est la peine pour vol simple ?', top_k=10)
warmup_s = time.perf_counter() - t0
print(f'  {len(_ids)} IDs in {_lat:.0f}ms (wall {warmup_s:.1f}s, first call includes model load)')
print(f'  First 5: {_ids[:5]}')
print(f'  VRAM after warmup: {_vram()}')

# Second call shows steady-state latency
_ids2, _lat2 = hybrid_retriever.retrieve('Quelles sont les conditions du contrat de travail ?', top_k=10)
print(f'  Steady-state latency: {_lat2:.0f}ms  (this is the per-query overhead for hybrid retrieval)')

In [ ]:
# ── Cell 11: TEST experiment (hybrid first stage, resumable) ──────────────────
# Experiment: llm_rerank_binary_top50_hybrid_rrf_k60_test
#
# Interrupt recovery:
#   - Score cache checkpointed every CACHE_CKPT_EVERY fresh questions → LLM calls never repeated
#   - Per-question checkpoint (.ckpt.json) → first-stage retrieval also skipped on resume
#   - Re-run this cell after interruption; it detects the checkpoint automatically
#
# Skips automatically if final result file already exists.
import json, os, sys, time
import numpy as np
from pathlib import Path

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import pandas as pd
from evaluation.split import load_questions
from evaluation.stratify import load_strata
from evaluation.runner import run_experiment, save_result
from retrieval.llm_reranker import LLMJudgeReranker
from retrieval.agentic.llm_client import OllamaClient

DB_PATH     = Path('output/bsard_corpus.db')
CORPUS_PATH = Path('output/bsard_articles_dedup.parquet')
RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH  = Path(f'{REPO_DIR}/output/llm_judge_cache_binary_test_tok1000.json')
TEST_JSON   = RESULTS_DIR / f'llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_hybrid_rrf_k60_test.json'
CKPT_PATH   = TEST_JSON.with_suffix('.ckpt.json')  # per-question resume checkpoint

CACHE_CKPT_EVERY = 20  # save cache + per-q checkpoint every N fresh questions

if TEST_JSON.exists():
    d  = json.loads(TEST_JSON.read_text())
    m  = d['metrics']
    hp = d.get('hyperparameters', {})
    print(f'{TEST_JSON.name} already exists — skipping.')
    print(f"  first_stage = {hp.get('first_stage')}")
    print(f"  R@10={m['Recall@10']:.4f}  R@100={m['Recall@100']:.4f}  MRR@10={m['MRR@10']:.4f}")
else:
    # ── Load data ──────────────────────────────────────────────────────────────
    print('Loading corpus and questions...')
    corpus        = pd.read_parquet(CORPUS_PATH)
    article_texts = dict(zip(corpus['article_id'], corpus['article_text']))
    questions     = load_questions(DB_PATH, subset='test')
    qid_map       = {q['question_text']: q['question_id'] for q in questions}
    gt            = {q['question_id']: q['relevant_article_ids'] for q in questions}
    strata        = load_strata()
    print(f'  {len(questions)} test questions | {len(article_texts):,} articles')

    # ── Build reranker ─────────────────────────────────────────────────────────
    llm_client = OllamaClient()
    reranker   = LLMJudgeReranker(
        first_stage_retriever=hybrid_retriever,
        article_texts=article_texts,
        llm_client=llm_client,
        top_n=BEST_TOP_N,
        max_article_tokens=1000,
        prompt_variant=BEST_VARIANT,
        cache_path=CACHE_PATH,
        question_id_fn=lambda q: qid_map.get(q, hash(q) & 0x7FFFFFFF),
        fewshot_path=Path('evaluation/data/fewshot_examples.json'),
    )

    # ── Load per-question checkpoint (interrupt resume) ─────────────────────────
    per_q_checkpoint = {}
    if CKPT_PATH.exists():
        per_q_checkpoint = json.loads(CKPT_PATH.read_text(encoding='utf-8'))
        n_ckpt = len(per_q_checkpoint)
        n_fresh_needed = len(questions) - n_ckpt
        print(f'\nCheckpoint loaded: {n_ckpt}/{len(questions)} questions done')
        print(f'  {n_fresh_needed} questions remaining — LLM calls served from score cache where possible')
    else:
        print(f'\nNo checkpoint — starting from question 1/{len(questions)}')

    # ── Monkey-patch retrieve() for checkpointing + progress stats ─────────────
    _fresh_count   = [0]
    _fresh_lats_ms = []
    _running_r10   = []
    _t0_run        = time.perf_counter()
    _orig_retrieve = reranker.retrieve

    def _retrieve_tracked(query: str, top_k: int = 100):
        qid     = qid_map.get(query, hash(query) & 0x7FFFFFFF)
        qid_str = str(qid)
        rel     = gt.get(qid, [])

        # Fast path: resume from per-question checkpoint (instant — no retrieval or LLM)
        if qid_str in per_q_checkpoint:
            ids = per_q_checkpoint[qid_str]
            if rel:
                _running_r10.append(len(set(ids[:10]) & set(rel)) / len(rel))
            return ids[:top_k], 0.0

        # Slow path: full hybrid retrieval + LLM re-ranking
        ids, lat = _orig_retrieve(query, top_k=top_k)
        per_q_checkpoint[qid_str] = ids
        _fresh_count[0] += 1
        _fresh_lats_ms.append(lat)

        if rel:
            _running_r10.append(len(set(ids[:10]) & set(rel)) / len(rel))

        n_done    = len(per_q_checkpoint)
        n_fresh   = _fresh_count[0]
        elapsed_s = time.perf_counter() - _t0_run
        avg_lat_s = sum(_fresh_lats_ms) / len(_fresh_lats_ms) / 1000
        eta_s     = avg_lat_s * max(0, len(questions) - n_done)
        r10_str   = f'{np.mean(_running_r10):.4f}' if _running_r10 else '-'

        # Periodic checkpoint save
        if n_fresh % CACHE_CKPT_EVERY == 0:
            reranker.save_cache()
            CKPT_PATH.write_text(
                json.dumps(per_q_checkpoint, ensure_ascii=False), encoding='utf-8'
            )
            stats_now  = reranker.get_stats()
            cache_size = stats_now.get('cache_size', '?')
            parse_fail = stats_now.get('parse_failure_rate', 0.0)
            print(
                f'[ckpt {n_done:3d}/{len(questions)}] '
                f'R@10={r10_str} | '
                f'lat={lat/1000:.1f}s  avg={avg_lat_s:.1f}s/q | '
                f'elapsed={elapsed_s/60:.1f}m  ETA≈{eta_s/60:.0f}m | '
                f'cache={cache_size:,}  parse_fail={parse_fail:.3f}',
                flush=True,
            )

        return ids, lat

    reranker.retrieve = _retrieve_tracked

    # ── Run experiment (run_experiment() provides its own tqdm progress bar) ────
    print(f'\nStarting experiment: llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_hybrid_rrf_k60_test')
    print(f'  Checkpoint save every {CACHE_CKPT_EVERY} fresh questions')
    print(f'  Score cache: {CACHE_PATH.name}')

    t_exp = time.perf_counter()
    try:
        result = run_experiment(
            retriever=reranker,
            questions=questions,
            experiment_id=f'llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_hybrid_rrf_k60_test',
            hyperparameters={
                'first_stage':           'hybrid_rrf_k60',
                'sparse_retriever':      'bm25_tuned_k11.5_b0.25',
                'dense_retriever':       'intfloat_multilingual_e5_large_concat_2x',
                'fusion_method':         'rrf',
                'rrf_k':                  60,
                'first_stage_k':          100,
                'llm_backbone':          'llama3.1:8b',
                'llm_temperature':        0.0,
                'top_n':                  BEST_TOP_N,
                'max_article_tokens':     1000,
                'scoring_method':        'llm_judge_binary',
                'prompt_variant':        BEST_VARIANT,
                'fewshot_examples_file': 'evaluation/data/fewshot_examples.json',
                'bm25_k1':                1.5,
                'bm25_b':                 0.25,
                'bm25_normalization':    'lemmatize',
                'bm25_field_weighting':  'text_only',
            },
            preprocessing={
                'normalization':    'lemmatize',
                'field_weighting':  'concat_2x',
                'embedding_prefix': 'query/passage',
            },
            strata=strata,
            top_k=100,
        )
    finally:
        # Always save on exit — even if interrupted by KeyboardInterrupt or error
        reranker.retrieve = _orig_retrieve
        reranker.save_cache()
        CKPT_PATH.write_text(
            json.dumps(per_q_checkpoint, ensure_ascii=False), encoding='utf-8'
        )
        print(
            f'\n[exit] Cache + checkpoint saved '
            f'({len(per_q_checkpoint)}/{len(questions)} questions done)'
        )

    exp_wall = time.perf_counter() - t_exp

    # ── Patch latency stats: resumed questions have lat=0 — report fresh-only ──
    if _fresh_lats_ms:
        arr = np.array(_fresh_lats_ms)
        result['latency_distribution_fresh_only'] = {
            'n_fresh_questions': len(arr),
            'mean_ms':  float(arr.mean()),
            'std_ms':   float(arr.std()),
            'p50_ms':   float(np.percentile(arr, 50)),
            'p90_ms':   float(np.percentile(arr, 90)),
            'note': 'Latency for questions not loaded from per-question checkpoint',
        }

    stats = reranker.get_stats()
    lbd   = reranker.get_latency_breakdown()
    result['total_experiment_wall_clock_s'] = round(exp_wall, 1)
    result['latency_breakdown_ms_mean'] = {
        'first_stage':  lbd.get('first_stage_ms_mean', 0.0),
        'llm_scoring':  lbd.get('llm_scoring_ms_mean', 0.0),
    }
    result['llm_rerank_stats'] = {
        'mean_llm_calls_per_query': BEST_TOP_N,
        'cache_size_after_run':     stats['cache_size'],
        'parse_failure_rate':       stats['parse_failure_rate'],
    }
    result['significance_vs_anchor'] = {
        'anchor_experiment_id': 'hybrid_rrf_k60_test',
        'p_value_recall10': None, 'significant': None,
        'note': 'Patched by Cell 12',
    }

    saved = save_result(result, results_dir=RESULTS_DIR)
    m     = result['metrics']

    print(f'\nSaved → {saved}')
    print(f"  R@10={m.get('Recall@10',0):.4f}  R@100={m.get('Recall@100',0):.4f}  "
          f"MRR@10={m.get('MRR@10',0):.4f}")
    print(f"  parse_failure_rate = {stats['parse_failure_rate']:.4f}")
    print(f'  Wall clock: {exp_wall/60:.1f} min  |  Fresh questions: {len(_fresh_lats_ms)}')

    # Clean up checkpoint only after full successful save
    if CKPT_PATH.exists():
        CKPT_PATH.unlink()
        print('  Per-question checkpoint cleaned up.')

In [ ]:
# ── Cell 12: Significance tests ───────────────────────────────────────────────
# Primary anchor:   hybrid_rrf_k60_test  → value of LLM re-ranking on hybrid pool
# Secondary anchor: llm_rerank_binary_top50_test (T4.0-BM25) → pool quality effect
import json, os, sys
import numpy as np
from pathlib import Path
from scipy.stats import ttest_rel
from bsard_evaluation import per_query_recall as _bsard_per_query_recall

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

RESULTS_DIR      = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')
HYBRID_T40_PATH  = RESULTS_DIR / f'llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_hybrid_rrf_k60_test.json'
HYBRID_BASE_PATH = Path(f'{REPO_DIR}/output/results/hybrid/hybrid_rrf_k60_test.json')
BM25_T40_PATH    = RESULTS_DIR / f'llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_test.json'

if not HYBRID_T40_PATH.exists():
    print(f'T4.0-hybrid result not found — run Cell 11 first.')
else:
    t40_hybrid = json.loads(HYBRID_T40_PATH.read_text(encoding='utf-8'))

    if '_trec_run' not in t40_hybrid or '_trec_qrels' not in t40_hybrid:
        raise RuntimeError('_trec_run/_trec_qrels missing from T4.0-hybrid result JSON')

    r_hyb40_k10  = _bsard_per_query_recall(t40_hybrid['_trec_qrels'], t40_hybrid['_trec_run'], 10)
    r_hyb40_k100 = _bsard_per_query_recall(t40_hybrid['_trec_qrels'], t40_hybrid['_trec_run'], 100)


    print(f'T4.0-hybrid  R@10 = {np.mean(r_hyb40_k10):.4f}  \''
          f'R@100 = {np.mean(r_hyb40_k100):.4f}')

    # ── Primary: T4.0-hybrid vs hybrid_rrf_k60 (T3-A) ─────────────────────────
    print('\n' + '='*60)
    print('PRIMARY: T4.0-hybrid vs hybrid_rrf_k60 (T3-A)')
    print('  Measures: value of LLM re-ranking on top of hybrid candidate pool')
    if HYBRID_BASE_PATH.exists():
        hyb_base     = json.loads(HYBRID_BASE_PATH.read_text(encoding='utf-8'))
        r_base_k10   = _bsard_per_query_recall(hyb_base['_trec_qrels'], hyb_base['_trec_run'], 10)
        r_base_k100  = _bsard_per_query_recall(hyb_base['_trec_qrels'], hyb_base['_trec_run'], 100)
        if r_base_k10 is not None:
            _, p10  = ttest_rel(r_hyb40_k10,  r_base_k10)
            _, p100 = ttest_rel(r_hyb40_k100, r_base_k100)
            delta10  = np.mean(r_hyb40_k10)  - np.mean(r_base_k10)
            delta100 = np.mean(r_hyb40_k100) - np.mean(r_base_k100)
            print(f'  T4.0-hybrid    R@10 = {np.mean(r_hyb40_k10):.4f}')
            print(f'  hybrid_rrf_k60 R@10 = {np.mean(r_base_k10):.4f}')
            print(f'  Delta R@10  = {delta10:+.4f}')
            print(f'  Delta R@100 = {delta100:+.4f}')
            sig_str = 'SIGNIFICANT (p<0.05)' if p10 < 0.05 else 'not significant'
            print(f'  p-value R@10  = {p10:.4f}  → {sig_str}')
            print(f'  p-value R@100 = {p100:.4f}')
            t40_hybrid['significance_vs_anchor'] = {
                'anchor_experiment_id': 'hybrid_rrf_k60_test',
                'p_value_recall10':  round(float(p10),  4),
                'p_value_recall100': round(float(p100), 4),
                'significant':       bool(p10 < 0.05),
            }
        else:
            print('  [WARN] per_query_recalls missing in hybrid_rrf_k60_test.json — significance skipped')
    else:
        print(f'  [WARN] {HYBRID_BASE_PATH.name} not found')
        print(f'         Upload results/hybrid/hybrid_rrf_k60_test.json to blob, re-run Cell 3')

    # ── Secondary: T4.0-hybrid vs T4.0-BM25 (pool quality effect) ────────────
    print('\nSECONDARY: T4.0-hybrid vs T4.0-BM25')
    print('  Measures: pool quality effect — same LLM re-ranker, different first stage')
    if BM25_T40_PATH.exists():
        bm25_t40     = json.loads(BM25_T40_PATH.read_text(encoding='utf-8'))
        r_bm25_k10   = _bsard_per_query_recall(bm25_t40['_trec_qrels'], bm25_t40['_trec_run'], 10)
        r_bm25_k100  = _bsard_per_query_recall(bm25_t40['_trec_qrels'], bm25_t40['_trec_run'], 100)
        if r_bm25_k10 is not None:
            _, p10s  = ttest_rel(r_hyb40_k10,  r_bm25_k10)
            _, p100s = ttest_rel(r_hyb40_k100, r_bm25_k100)
            print(f'  T4.0-hybrid R@10 = {np.mean(r_hyb40_k10):.4f}')
            print(f'  T4.0-BM25   R@10 = {np.mean(r_bm25_k10):.4f}')
            print(f'  Delta R@10  = {np.mean(r_hyb40_k10) - np.mean(r_bm25_k10):+.4f}')
            sig_str = 'SIGNIFICANT (p<0.05)' if p10s < 0.05 else 'not significant'
            print(f'  p-value R@10  = {p10s:.4f}  → {sig_str}')
            t40_hybrid.setdefault('secondary_significance', {})
            t40_hybrid['secondary_significance']['vs_llm_rerank_binary_top50_bm25_test'] = {
                'anchor_experiment_id': f'llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_test',
                'p_value_recall10':  round(float(p10s),  4),
                'p_value_recall100': round(float(p100s), 4),
                'significant':       bool(p10s < 0.05),
            }
        else:
            print('  [WARN] per_query_recalls missing in T4.0-BM25 result — skipped')
    else:
        print(f'  [WARN] {BM25_T40_PATH.name} not found — skipped')

    HYBRID_T40_PATH.write_text(json.dumps(t40_hybrid, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'\nPatched {HYBRID_T40_PATH.name} with significance results.')

In [ ]:
# ── Cell 13: Final results summary ────────────────────────────────────────────
import json
from pathlib import Path

results_dir = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')

print(f'{"Experiment":<52}  R@10    R@100   MRR@10  sig      p-val')
print('-' * 90)
for f in sorted(results_dir.glob('llm_rerank_*.json')):
    d   = json.loads(f.read_text())
    m   = d['metrics']
    sig = d.get('significance_vs_anchor', {}).get('significant', '-')
    p   = d.get('significance_vs_anchor', {}).get('p_value_recall10', None)
    p_str  = f'p={p:.4f}' if p is not None else ''
    anc    = d.get('significance_vs_anchor', {}).get('anchor_experiment_id', '')
    print(f"  {d['experiment_id']:<50}  {m.get('Recall@10',0):.4f}  "
          f"{m.get('Recall@100',0):.4f}  {m.get('MRR@10',0):.4f}  {str(sig):<8} {p_str}")
    if anc:
        print(f"    anchor: {anc}")

In [ ]:
# ── Cell 14: Upload results to Azure Blob Storage ─────────────────────────────
# Uploads all result JSONs in llm_rerank/ + updated score cache.
# output/ is gitignored — results are NOT committed to git.
from pathlib import Path
from azure.storage.blob import ContainerClient
from tqdm.auto import tqdm

UPLOAD_MAP = {
    f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank': 'results/agentic/llm_judge/llm_rerank',
}

client   = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)
uploaded = []

for local_dir, blob_prefix in UPLOAD_MAP.items():
    local_path = Path(local_dir)
    if not local_path.exists():
        print(f'  [{blob_prefix}] directory not found — skipping.')
        continue
    json_files = sorted(local_path.glob('*.json'))
    for json_file in tqdm(json_files, desc=f'  {blob_prefix}', unit='file'):
        blob_name = f'{blob_prefix}/{json_file.name}'
        with open(json_file, 'rb') as f:
            client.get_blob_client(blob_name).upload_blob(f, overwrite=True)
        uploaded.append(blob_name)

# Upload updated score cache
cache_path = Path(f'{REPO_DIR}/output/llm_judge_cache_binary_test_tok1000.json')
if cache_path.exists():
    size_mb = cache_path.stat().st_size / 1e6
    print(f'  Uploading score cache ({size_mb:.1f} MB) ...', end='', flush=True)
    with open(cache_path, 'rb') as f:
        client.get_blob_client('llm_judge_cache_binary_test_tok1000.json').upload_blob(f, overwrite=True)
    print(' done')
    uploaded.append('llm_judge_cache_binary_test_tok1000.json')

print(f'\nUploaded {len(uploaded)} file(s) to blob storage.')